In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("datasets/dataset_final.csv")

In [3]:
df.columns

Index(['Domain_URL_Ratio', 'Count_www', 'Count_/', 'Path_Length', 'URL/Path',
       'Character_Repetition', 'Having_Path', 'Special_Char_Alphabet_Ratio',
       'ShannonEntropy', 'fd_length', 'Digit/Letter', 'Count_Digit',
       'Kolmogorov_Complexity', 'Average_Word', 'FractalDimension',
       'Count_Letter', 'Vowel/Consonant', 'Count_Dot', 'Base64_Pattern_Cnt',
       'Host_Precense_Of_Digit', 'Domain_Length_Of_URL', 'Subdomain',
       'Count_-', 'Uppercase_Lowercase_Ratio', 'Longest_Word_in_Hostname',
       'Count_Embed_Domain', 'Tld_Length', 'use_of_ip_address', 'Longest_Word',
       'Count_?', 'Query_Length', 'Count_&', 'Count_;', 'Port_Number',
       'Label'],
      dtype='object')

In [4]:
import pandas as pd
import time
import psutil

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.base import clone

# ================= DATA =================
X = df.drop('Label', axis=1)
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ================= MODELLER =================
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(),
    "Linear SVM": LinearSVC(),  # ✅ değişti
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# ================= HİBRİT MODEL =================
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss')),
        ('svm', LinearSVC())  # ✅ burada da değişti
    ],
    final_estimator=LogisticRegression(),
    n_jobs=-1
)

models["Stacking (Hybrid)"] = stacking_model

# ================= EĞİTİM =================
results = []
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
process = psutil.Process()

for name, model in models.items():
    try:
        print(f">>> {name} eğitiliyor...")

        current_model = clone(model)

        start_time = time.time()
        start_mem = process.memory_info().rss / (1024 ** 2)

        current_model.fit(X_train, y_train)
        train_time = time.time() - start_time

        pred_start = time.time()
        y_pred = current_model.predict(X_test)
        pred_time = time.time() - pred_start

        end_mem = process.memory_info().rss / (1024 ** 2)
        mem_used = end_mem - start_mem
        cpu_used = psutil.cpu_percent(interval=0.1)

        row = {
            "Model": name,
            "Test_Accuracy": accuracy_score(y_test, y_pred),
            "Test_Precision": precision_score(y_test, y_pred),
            "Test_Recall": recall_score(y_test, y_pred),
            "Test_F1": f1_score(y_test, y_pred),
            "Train_Time (s)": round(train_time, 4),
            "Predict_Time (s)": round(pred_time, 4),
            "Memory_Usage (MB)": round(mem_used, 2),
            "CPU_Usage (%)": cpu_used
        }

        # ROC-AUC
        try:
            if hasattr(current_model, "predict_proba"):
                y_prob = current_model.predict_proba(X_test)[:, 1]
            else:
                y_prob = current_model.decision_function(X_test)
            row["Test_ROC_AUC"] = roc_auc_score(y_test, y_prob)
        except:
            row["Test_ROC_AUC"] = None

        print(f"    {name} için Cross-Validation...")
        cv_scores = cross_validate(clone(model), X, y, cv=5, scoring=scoring, n_jobs=-1)

        row["CV_Accuracy"] = cv_scores["test_accuracy"].mean()
        row["CV_F1"] = cv_scores["test_f1"].mean()
        row["CV_ROC_AUC"] = cv_scores["test_roc_auc"].mean()

        results.append(row)
        print(f"--- {name} tamamlandı ---\n")

    except Exception as e:
        print(f"!!! {name} hatası: {e}")

# ================= SONUÇ =================
results_table = pd.DataFrame(results).sort_values("Test_Accuracy", ascending=False)
results_table = results_table.reset_index(drop=True)
results_table.index = results_table.index + 1

print("\n=== TÜM MODELLER PERFORMANS TABLOSU ===")
print(results_table)

>>> Logistic Regression eğitiliyor...
    Logistic Regression için Cross-Validation...
--- Logistic Regression tamamlandı ---

>>> Random Forest eğitiliyor...
    Random Forest için Cross-Validation...
--- Random Forest tamamlandı ---

>>> Decision Tree eğitiliyor...
    Decision Tree için Cross-Validation...
--- Decision Tree tamamlandı ---

>>> Linear SVM eğitiliyor...
    Linear SVM için Cross-Validation...
--- Linear SVM tamamlandı ---

>>> Naive Bayes eğitiliyor...
    Naive Bayes için Cross-Validation...
--- Naive Bayes tamamlandı ---

>>> KNN eğitiliyor...
    KNN için Cross-Validation...
--- KNN tamamlandı ---

>>> Gradient Boosting eğitiliyor...
    Gradient Boosting için Cross-Validation...
--- Gradient Boosting tamamlandı ---

>>> XGBoost eğitiliyor...
    XGBoost için Cross-Validation...
--- XGBoost tamamlandı ---

>>> Stacking (Hybrid) eğitiliyor...


[15:20:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.



    Stacking (Hybrid) için Cross-Validation...


[15:20:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:35] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

[15:20:53] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" }

--- Stacking (Hybrid) tamamlandı ---


=== TÜM MODELLER PERFORMANS TABLOSU ===
                 Model  Test_Accuracy  Test_Precision  Test_Recall   Test_F1  \
1              XGBoost       0.997788        0.999572     0.996189  0.997878   
2    Stacking (Hybrid)       0.997743        0.999429     0.996246  0.997835   
3        Random Forest       0.997327        0.998859     0.996018  0.997437   
4  Logistic Regression       0.997060        0.999372     0.994994  0.997178   
5        Decision Tree       0.996615        0.997635     0.995876  0.996755   
6    Gradient Boosting       0.996184        0.999513     0.993174  0.996334   
7                  KNN       0.991700        0.996755     0.987315  0.992013   
8           Linear SVM       0.942272        0.993685     0.895111  0.941826   
9          Naive Bayes       0.920713        0.995448     0.852024  0.918168   

   Train_Time (s)  Predict_Time (s)  Memory_Usage (MB)  CPU_Usage (%)  \
1          0.4251            0.0125            